# Transfer Learning for Nigerian Currency Recognition using ResNet18

## Overview

Deep learning models often require large amounts of labeled data and significant computational resources to learn effective visual representations from scratch. Transfer learning addresses this challenge by adapting models that have already learned rich feature representations from large-scale image datasets.

This notebook implements a transfer learning approach for Nigerian currency denomination recognition using **ResNet18**, a convolutional neural network pretrained on the ImageNet dataset. Instead of learning visual features from randomly initialized weights, the pretrained network provides generalized feature extractors which are subsequently adapted to classify Nigerian currency denominations.

The objective of this experiment is to evaluate whether transfer learning can improve classification performance compared to the previously developed baseline convolutional neural network while reducing training time and improving model generalization.

## Import Required Libraries

The implementation relies on PyTorch and Torchvision for deep learning operations, Scikit-learn for model evaluation, Matplotlib for visualization, and several utility libraries for dataset management and experiment reproducibility.

In [9]:
# =============================================================================
# IMPORT REQUIRED LIBRARIES
# =============================================================================

import copy
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## Experiment Configuration

The experimental settings defined in this section establish the dataset locations, model hyperparameters, image dimensions, and other configuration variables that remain constant throughout the training process. Centralizing these parameters improves reproducibility and simplifies future experimentation.

In [10]:
# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================

DATASET_PATH = Path(r"E:/Merged V1")

TRAIN_PATH = DATASET_PATH / "Train"
VAL_PATH = DATASET_PATH / "Val"
TEST_PATH = DATASET_PATH / "Test"

IMAGE_SIZE = (224, 224)

BATCH_SIZE = 16

NUM_EPOCHS = 10

LEARNING_RATE = 1e-4

RANDOM_STATE = 42

MODEL_NAME = "ResNet18_TransferLearning"

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

## Computing Device Configuration

Deep learning models benefit significantly from hardware acceleration. This experiment automatically selects a CUDA-enabled GPU when available; otherwise, computation is performed using the CPU. This approach ensures portability across different computing environments.

In [11]:
# =============================================================================
# DEVICE CONFIGURATION
# =============================================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 50)
print(f"Device : {device}")

if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

print("=" * 50)

Device : cpu


## Image Transformations

Prior to model training, input images are resized to the dimensions expected by ResNet18 and converted into tensor representations. The training dataset additionally undergoes data augmentation to improve model robustness by exposing the network to realistic variations such as random horizontal flipping and slight rotations.

Finally, pixel values are normalized using the ImageNet mean and standard deviation, ensuring compatibility with the pretrained model.

In [12]:
# =============================================================================
# IMAGE TRANSFORMATIONS
# =============================================================================

train_transform = transforms.Compose([

    transforms.Resize(IMAGE_SIZE),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[0.485, 0.456, 0.406],

        std=[0.229, 0.224, 0.225]

    )

])

test_transform = transforms.Compose([

    transforms.Resize(IMAGE_SIZE),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[0.485, 0.456, 0.406],

        std=[0.229, 0.224, 0.225]

    )

])

## Dataset Loading

The merged dataset is organized into separate training, validation, and testing directories. Each subset is loaded independently using the `ImageFolder` utility provided by Torchvision, which automatically assigns class labels based on the folder structure.

The training subset applies data augmentation to improve generalization, while the validation and testing subsets retain only the essential preprocessing operations to ensure unbiased model evaluation.

In [16]:
# =============================================================================
# LOAD DATASETS
# =============================================================================

train_dataset = datasets.ImageFolder(
    root=TRAIN_PATH,
    transform=train_transform
)

validation_dataset = datasets.ImageFolder(
    root=VAL_PATH,
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    root=TEST_PATH,
    transform=test_transform
)

class_names = train_dataset.classes

print("=" * 60)
print(f"Number of Classes : {len(class_names)}")
print(f"Training Images   : {len(train_dataset)}")
print(f"Validation Images : {len(validation_dataset)}")
print(f"Testing Images    : {len(test_dataset)}")
print("=" * 60)

print("\nClasses")

for idx, class_name in enumerate(class_names):

    
    
    print(f"{idx}: ₦{class_name}")

Number of Classes : 8
Training Images   : 1815
Validation Images : 389
Testing Images    : 390

Classes
0: ₦10 naira
1: ₦100 naira
2: ₦1000 naira
3: ₦20 naira
4: ₦200 naira
5: ₦5 naira
6: ₦50 naira
7: ₦500 naira


## DataLoader Construction

The datasets are wrapped using PyTorch DataLoaders to enable efficient mini-batch processing during model training and evaluation. Training samples are shuffled before each epoch to improve learning stability, whereas validation and testing samples retain their original order to ensure consistent evaluation.

In [17]:
# =============================================================================
# CREATE DATALOADERS
# =============================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("DataLoaders created successfully.")

DataLoaders created successfully.


## Model Construction

A pretrained **ResNet18** model is adopted as the backbone for the currency recognition task. The convolutional layers retain the feature representations learned from the ImageNet dataset, while the original classification layer is replaced with a new fully connected layer corresponding to the eight Nigerian currency denominations.

To reduce computational cost and training time, the pretrained feature extractor is frozen during the initial training phase, allowing only the final classification layer to be updated.

In [18]:
# =============================================================================
# BUILD TRANSFER LEARNING MODEL
# =============================================================================

# Load pretrained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze pretrained layers
for param in model.parameters():
    param.requires_grad = False

# Replace classifier
model.fc = nn.Linear(
    model.fc.in_features,
    len(class_names)
)

model = model.to(device)

print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\CURRENT COMPUTERS/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [01:01<00:00, 757kB/s] 


Linear(in_features=512, out_features=8, bias=True)


## Training Configuration

The model is optimized using the Adam optimizer, while the Cross-Entropy Loss function is employed for multi-class classification. Since only the newly added classification layer is trainable, the optimizer updates only its parameters.

In [19]:
# =============================================================================
# TRAINING CONFIGURATION
# =============================================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=LEARNING_RATE
)

## Model Training

The model is trained using mini-batch gradient descent while monitoring its performance on the validation dataset after each epoch. The model parameters that achieve the highest validation accuracy are retained and saved for subsequent evaluation.

In [28]:
# =============================================================================
# TRAINING FUNCTION WITH CHECKPOINTING
# =============================================================================

def train_model(
    model,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    device,
    epochs
):

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_accuracy = 0.0
    start_epoch = 0

    # ----------------------------------------------------
    # Resume from checkpoint if available
    # ----------------------------------------------------

    if CHECKPOINT_PATH.exists():

        checkpoint = torch.load(
            CHECKPOINT_PATH,
            map_location=device
        )

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        history = checkpoint["history"]

        best_accuracy = checkpoint["best_accuracy"]

        start_epoch = checkpoint["epoch"] + 1

        print("=" * 60)
        print(f"Checkpoint found.")
        print(f"Resuming from Epoch {start_epoch+1}")
        print("=" * 60)

    else:

        print("=" * 60)
        print("No checkpoint found.")
        print("Starting new training...")
        print("=" * 60)

    # ----------------------------------------------------
    # TRAIN
    # ----------------------------------------------------

    for epoch in range(start_epoch, epochs):

        print(f"\nEpoch {epoch+1}/{epochs}")

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        train_bar = tqdm(
            train_loader,
            desc="Training",
            leave=False
        )

        for images, labels in train_bar:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item() * images.size(0)

            _, predicted = outputs.max(1)

            correct += predicted.eq(labels).sum().item()

            total += labels.size(0)

            train_bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        train_loss = running_loss / total
        train_acc = correct / total

        # ---------------- Validation ----------------

        model.eval()

        running_loss = 0.0
        correct = 0
        total = 0

        val_bar = tqdm(
            validation_loader,
            desc="Validation",
            leave=False
        )

        with torch.no_grad():

            for images, labels in val_bar:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)

                _, predicted = outputs.max(1)

                correct += predicted.eq(labels).sum().item()

                total += labels.size(0)

                val_bar.set_postfix(
                    loss=f"{loss.item():.4f}"
                )

        val_loss = running_loss / total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        # ----------------------------------------------------
        # Save Best Model
        # ----------------------------------------------------

        if val_acc > best_accuracy:

            best_accuracy = val_acc

            torch.save(
                model.state_dict(),
                BEST_MODEL_PATH
            )

            print("✓ Best model updated.")

        # ----------------------------------------------------
        # Save Checkpoint
        # ----------------------------------------------------

        torch.save({

            "epoch": epoch,

            "model_state_dict": model.state_dict(),

            "optimizer_state_dict": optimizer.state_dict(),

            "history": history,

            "best_accuracy": best_accuracy

        }, CHECKPOINT_PATH)

        print("✓ Checkpoint saved.")

    # ----------------------------------------------------
    # Load Best Model
    # ----------------------------------------------------

    model.load_state_dict(
        torch.load(
            BEST_MODEL_PATH,
            map_location=device
        )
    )

    return model, history

## Checkpoint Configuration

To prevent the loss of training progress due to unexpected interruptions, model checkpoints are saved after each epoch. These checkpoints contain the model parameters, optimizer state, current epoch, training history, and the best validation accuracy achieved.

If training is interrupted, the checkpoint is automatically loaded and training resumes from the last completed epoch.

In [27]:
# =============================================================================
# CHECKPOINT CONFIGURATION
# =============================================================================

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / "resnet18_checkpoint.pth"

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_resnet18.pth"

## Model Training Execution

The transfer learning model is trained using the predefined hyperparameters. Throughout the training process, the model with the highest validation accuracy is retained for subsequent evaluation.

In [29]:
from tqdm.auto import tqdm

In [31]:
# =============================================================================
# TRAIN MODEL
# =============================================================================

print("=" * 60)
print("Starting ResNet18 Transfer Learning...")
print("=" * 60)

model, history = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=NUM_EPOCHS
)

print("=" * 60)
print("Training Completed Successfully!")
print("=" * 60)

Starting ResNet18 Transfer Learning...
No checkpoint found.
Starting new training...

Epoch 1/10


Training:   0%|          | 0/114 [00:00<?, ?it/s]

Validation:   0%|          | 0/25 [00:00<?, ?it/s]

Train Loss: 1.9958 | Train Acc: 0.2347 | Val Loss: 1.8887 | Val Acc: 0.3393
✓ Best model updated.
✓ Checkpoint saved.

Epoch 2/10


Training:   0%|          | 0/114 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/25 [00:00<?, ?it/s]

Train Loss: 1.0089 | Train Acc: 0.7713 | Val Loss: 0.9125 | Val Acc: 0.8252
✓ Best model updated.
✓ Checkpoint saved.

Epoch 10/10


Training:   0%|          | 0/114 [00:00<?, ?it/s]

Validation:   0%|          | 0/25 [00:00<?, ?it/s]

Train Loss: 0.9682 | Train Acc: 0.7719 | Val Loss: 0.8670 | Val Acc: 0.8278
✓ Best model updated.
✓ Checkpoint saved.
Training Completed Successfully!


In [32]:
# =============================================================================
# SAVE MODEL
# =============================================================================

torch.save(
    model.state_dict(),
    f"{MODEL_NAME}.pth"
)

print(f"Model saved as {MODEL_NAME}.pth")

Model saved as ResNet18_TransferLearning.pth
